# AMD/NVIDIA 原始收入指引口径机械修复

首轮误读全年/分部收入及其他费用金额；仅改同份公告收入句边界和本季收入识别，保留未知与异常，零市场结果。

In [1]:
import json,hashlib,re
from pathlib import Path
from collections import Counter
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_amd_nvda_guidance_breadth_source_20261023';plan=json.loads((card/'plan.json').read_text());repair=json.loads((card/'numeric_parser_repair_plan.json').read_text());parent=root/plan['parent']['source_card'];raw=root/'data/snapshots/stage1_amd_nvda_sec_guidance_original_probe_20261023';sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert repair['frozen'] and sha(card/'plan.json')==repair['original_plan_sha256'] and sha(card/'numeric_attempt1_result.json')==repair['failed_attempt1_result_sha256'] and not (card/'numeric_repair_result.json').exists()
source=json.loads((parent/'quarter_event_adjudication_result.json').read_text());assert len(source['selected_events'])==126
words={'zero':0,'one':1,'two':2,'three':3,'four':4,'five':5,'six':6,'seven':7,'eight':8,'nine':9,'ten':10,'eleven':11,'twelve':12,'thirteen':13,'fourteen':14,'fifteen':15,'sixteen':16,'seventeen':17,'eighteen':18,'nineteen':19,'twenty':20}
def num(token):
 token=token.lower().strip();return float(token.replace(',','')) if re.fullmatch(r'[\d,.]+',token) else float(words[token]) if token in words else None
def usd_m(token,unit):return num(token)*(1000 if unit.lower().startswith('billion') else 1)
def release_text(body):
 parts=[]
 for doc in re.findall(rb'<DOCUMENT>(.*?)</DOCUMENT>',body,re.S|re.I):
  typ=re.search(rb'<TYPE>\s*([^\r\n<]+)',doc,re.I);content=re.search(rb'<TEXT>(.*?)</TEXT>',doc,re.S|re.I)
  if typ and content and typ.group(1).strip().upper() in (b'EX-99.1',b'EX-95.1'):
   parts.append(re.sub(r'\s+',' ',BeautifulSoup(content.group(1).decode('utf8','replace'),'html.parser').get_text(' ',strip=True)))
 return ' '.join(parts)
def guidance_clause(text):
 options=[]
 for marker in re.finditer(r'\b(?:current\s+)?outlook\b',text,re.I):
  segment=text[marker.end():marker.end()+2000]
  found=re.search(r'(?:Revenue\s+is\s+expected|AMD\s+expects\s+revenue|expects\s+revenue\s+to|Revenue\s+to\s+be)',segment,re.I)
  if found:options.append((found.start(),marker.end()+found.start()))
 if not options:return None,None
 _,position=min(options)
 raw=text[position:position+420]
 bounded=re.split(r'(?<!\d)\.\s+(?=[A-Z])|[•●·]',raw,maxsplit=1)[0].strip()
 return bounded,position
def current_quarter_revenue(text,prior):
 # Every pattern names the just-reported quarter; do not use annual or segment revenue.
 amount=r'\$\s*([\d,.]+)\s*(million|billion)'
 patterns=[
  r'\brevenue\s+for\s+the\s+(?:first|second|third|fourth)\s+quarter.{0,110}?\b(?:of|was)\s+'+amount,
  r'\b(?:first|second|third|fourth)\s+quarter\s+revenue\s+(?:was|of)\s+'+amount,
  r'\brevenue\s+in\s+Q[1-4]\s+(?:of|was)\s+'+amount,
  r'\bquarterly\s+revenue.{0,85}?\bto\s+'+amount,
  r'\breported\s+revenue\s+(?:of|was)\s+'+amount,
  r'\brevenue\s+(?:of|was)\s+'+amount]
 for idx,pattern in enumerate(patterns):
  match=re.search(pattern,prior,re.I)
  if match:
   value=usd_m(match.group(1),match.group(2))
   if value and value>0:return value,idx,match.group(0)[:220]
 return None,None,None
rows=[]
for event in source['selected_events']:
 acc=event['accession'];p=raw/f'item202_{acc}.txt';assert sha(p)==event['raw_sha256'];text=release_text(p.read_bytes());issuer=event['issuer']
 if event['revenue_outlook']=='qualitative_unknown':
  rows.append({'issuer':issuer,'filing_date':event['filing_date'],'accession':acc,'raw_sha256':event['raw_sha256'],'status':'qualitative_unknown','growth':None,'format':None,'reason':'source_adjudicated_qualitative','guidance_excerpt':None});continue
 clause,position=guidance_clause(text);status='unresolved';reason='no_supported_revenue_sentence';growth=None;format_=None;reported=None;forecast=None;reported_excerpt=None
 if clause:
  # Clause is bounded to the revenue sentence/bullet; expenses in later bullets cannot be forecast revenue.
  between=re.search(r'between\s+\$?\s*([\d,.]+)\s*(million|billion)?\s+and\s+\$?\s*([\d,.]+)\s*(million|billion)',clause,re.I)
  absolute=re.search(r'\$\s*([\d,.]+)\s*(million|billion)',clause,re.I)
  if between:
   forecast=(usd_m(between.group(1),between.group(2) or between.group(4))+usd_m(between.group(3),between.group(4)))/2;format_='absolute_range'
  elif absolute:
   forecast=usd_m(absolute.group(1),absolute.group(2));format_='absolute_central'
  if forecast is not None:
   reported,pattern_index,reported_excerpt=current_quarter_revenue(text,text[:position]);
   if reported is not None:
    growth=forecast/reported-1;status='numeric';reason='absolute_guidance_over_explicit_current_quarter_revenue'
   else:reason='explicit_current_quarter_revenue_not_parseable'
  if growth is None and forecast is None:
   pct=re.search(r'\b(up|increase|decrease|down)\s+(?:approximately\s+)?([\d,.]+|[a-z]+)(?:\s+to\s+([\d,.]+|[a-z]+))?\s*(?:percent|%)',clause,re.I)
   if pct:
    low=num(pct.group(2));high=num(pct.group(3)) if pct.group(3) else low
    if low is not None and high is not None:
     sign=-1 if pct.group(1).lower() in ('decrease','down') else 1;growth=sign*(low+high)/200;status='numeric';format_='direct_sequential_percent';reason='explicit_numeric_sequential_percent'
   elif re.search(r'\b(?:relatively\s+)?flat\b.{0,35}(?:plus\s+or\s+minus|\+/-)\s+(?:two|2)\s*(?:percent|%)',clause,re.I):
    growth=0.0;status='numeric';format_='direct_flat_plus_minus_percent';reason='explicit_numeric_symmetric_flat_guidance'
  if growth is None and reason=='no_supported_revenue_sentence':reason='bounded_revenue_sentence_has_no_supported_numeric_form'
 if growth is not None and not (-1<growth<3):status='unresolved';reason='source_ratio_outside_broad_semantic_bounds';growth=None
 rows.append({'issuer':issuer,'filing_date':event['filing_date'],'accession':acc,'raw_sha256':event['raw_sha256'],'status':status,'format':format_,'growth':growth,'forecast_next_mid_USD_million':forecast,'reported_current_USD_million':reported,'reported_current_excerpt':reported_excerpt,'guidance_excerpt':clause[:270] if clause else None,'reason':reason})
assert len(rows)==126
by={issuer:{'numeric':sum(x['issuer']==issuer and x['status']=='numeric' for x in rows),'unknown':sum(x['issuer']==issuer and x['status']!='numeric' for x in rows)} for issuer in ('AMD','NVDA')};outliers=[{'issuer':x['issuer'],'filing_date':x['filing_date'],'accession':x['accession'],'growth':x['growth'],'reported':x.get('reported_current_USD_million'),'forecast':x.get('forecast_next_mid_USD_million')} for x in rows if x['growth'] is not None and abs(x['growth'])>0.6]
result={'status':'source_numeric_semantics_repair_attempt2_partial_until_all_unresolved_and_outliers_adjudicated','plan_sha256':sha(card/'numeric_parser_repair_plan.json'),'parent_source_plan_sha256':sha(card/'plan.json'),'rows':rows,'by_issuer':by,'unresolved':[{'issuer':x['issuer'],'filing_date':x['filing_date'],'accession':x['accession'],'reason':x['reason'],'guidance_excerpt':x['guidance_excerpt'],'reported_current_excerpt':x.get('reported_current_excerpt')} for x in rows if x['status']!='numeric'],'large_source_ratio_review_rows':outliers,'new_GETs':0,'Qlib_calls':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'ready_for_Qlib_map':False};(card/'numeric_repair_result.json').write_text(json.dumps(result,indent=2)+'\n');print({'status':result['status'],'by_issuer':by,'unresolved':len(result['unresolved']),'large_ratio_review':len(outliers),'formats':dict(Counter(x['format'] for x in rows if x['format']))})


{'status': 'source_numeric_semantics_repair_attempt2_partial_until_all_unresolved_and_outliers_adjudicated', 'by_issuer': {'AMD': {'numeric': 61, 'unknown': 2}, 'NVDA': {'numeric': 63, 'unknown': 0}}, 'unresolved': 2, 'large_ratio_review': 3, 'formats': {'direct_sequential_percent': 29, 'absolute_central': 90, 'direct_flat_plus_minus_percent': 1, 'absolute_range': 4}}


## 中文结论

第二轮限定下一季收入句与明确的本季收入语境，剩下三个 AMD 分母异常和两期定性未知。 见 `conclusion.md`。